## Week 1 primer: the language you need for the live coding

Computational Macro (HWS 2026), University of Mannheim

The course solves models in class, in notebooks that we fill in together. This
primer introduces the language constructs those notebooks use, and nothing
else, using the growth model of session 1 as the running example.

**Before you start.** Open this notebook in VS Code and pick the `compmacro` kernel in the top
right corner. That is the course environment, which pins the package versions:
`Python/README.md` explains how to create it from `environment.yml` with
`conda env create -f environment.yml`. If `import numpy` in the first cell
fails, the wrong kernel is selected.

**The model** (session 1, "The same example by dynamic programming"):

$$
v(k) = \max_{0 \leq k' \leq k^\alpha} \bigl\{ \ln(k^\alpha - k') + \beta \, v(k') \bigr\},
\qquad \alpha = 0.3, \quad \beta = 0.96 .
$$

**What we derived** (guess and verify, and "VFI by hand"):

$$
v(k) = E + F \ln k, \quad F = \frac{\alpha}{1 - \alpha\beta}, \qquad
k' = \alpha\beta \, k^\alpha, \qquad
F_{n+1} = \alpha + \alpha\beta \, F_n \ \xrightarrow{n \to \infty} \ F .
$$

Five short parts: arrays, containers, functions, loops, plots. Run every cell
with Shift+Enter and change things to see what happens.

In [ ]:
from dataclasses import dataclass

import numpy as np
import matplotlib.pyplot as plt

### 1. Arrays and elementwise arithmetic

A function evaluated on a grid is a vector: $N$ numbers, one per grid point. The
first object we need is the grid itself, $\mathcal{K} = \{k_1, \ldots, k_N\}$, then
output $k_i^\alpha$ and the closed form policy $\alpha\beta k_i^\alpha$ at every
point. In Python the tool is NumPy: `np.linspace` builds evenly spaced points, and
arithmetic on a NumPy array is elementwise by default, `k ** alpha`. NumPy
functions such as `np.log` are elementwise too.

In [ ]:
alpha, beta = 0.3, 0.96
kstar = (alpha * beta) ** (1 / (1 - alpha))  # steady state of the closed form policy
k = np.linspace(0.1 * kstar, 2.0 * kstar, 10)   # a small grid to look at
output = k ** alpha                          # k_i^alpha at every grid point, elementwise
policy = alpha * beta * output               # alpha beta k_i^alpha, the closed form policy
np.column_stack([k, output, policy])         # three columns side by side

A second way to place points: equidistant in $\ln k$ rather than in $k$. Take
the logs of the bounds, space them evenly, and map back with the exponential.
The points are denser at low $k$.

In [ ]:
k_log = np.exp(np.linspace(np.log(0.1 * kstar), np.log(2.0 * kstar), 10))
print(f"equidistant, spacing at the start and at the end: {k[1] - k[0]:.4f}  {k[-1] - k[-2]:.4f}")
print(f"log spaced,  spacing at the start and at the end: {k_log[1] - k_log[0]:.4f}  {k_log[-1] - k_log[-2]:.4f}")

### 2. Containers for parameters and grids

Parameters live in a container with named fields, one for the economics and one
for the numerics, so that a change to the grid can never touch $\alpha$ or
$\beta$ by accident. A frozen `dataclass` defines such a container with default values. It is
created with keyword arguments, and any field can be overridden at creation,
`NumericalParameters(nk=30)`. Fields are read with a dot, `par.alpha`. Frozen
means the fields cannot be changed afterwards, which is what we want for
parameters. The grid gets a small dataclass of its own.

In [ ]:
@dataclass(frozen=True)
class EconomicParameters:
    alpha: float = 0.3      # capital share
    beta: float = 0.96      # discount factor


@dataclass(frozen=True)
class NumericalParameters:
    nk: int = 50            # number of grid points
    crit: float = 1e-8      # convergence tolerance
    maxiter: int = 1000     # iteration cap


@dataclass(frozen=True)
class Grids:
    k: np.ndarray           # the capital grid


par = EconomicParameters()
mpar = NumericalParameters(nk=40)            # override one default, keep the others
gri = Grids(k=np.linspace(0.1 * kstar, 2.0 * kstar, mpar.nk))
print(par)
print(mpar)
print("grid with", len(gri.k), "points, read with gri.k")

### 3. Functions

A function takes arguments and returns a value. In Python a function is written with `def name(x):` followed by an indented
body that ends with `return`. For a one-line function there is also the
`lambda` form, `name = lambda x: expression`. Both are called the same way,
`name(2.0)`.

Three functions from session 1: the utility function $u(c) = \ln c$, the closed
form value function $v(k) = E + F \ln k$, and the policy $k' = \alpha\beta k^\alpha$.
The constants $E$ and $F$ are computed once from the parameters and used inside
the functions.

In [ ]:
def util(c):                                 # def, an indented body, return
    return np.log(c)


F_star = par.alpha / (1 - par.alpha * par.beta)
E_star = (np.log(1 - par.alpha * par.beta) + par.beta * F_star * np.log(par.alpha * par.beta)) / (1 - par.beta)


def v_true(k):
    return E_star + F_star * np.log(k)


policy_true = lambda k: par.alpha * par.beta * k ** par.alpha   # the one-line form

print(f"u(1) = {util(1.0):.3f}, u(2) = {util(2.0):.3f}, v(k*) = {v_true(kstar):.4f}, k'(k*) = {policy_true(kstar):.4f}")

Functions applied to a vector give a vector, so evaluating the closed form on
the whole grid is one line.


In [ ]:
v_grid = v_true(gri.k)                       # an array in, an array out
policy_grid = policy_true(gri.k)
np.column_stack([gri.k[:5], v_grid[:5], policy_grid[:5]])

### 4. Loops

A `while` loop repeats its body as long as a condition holds. The condition can
be anything: a counter below a limit, an input not yet received, or, the case we
care about, a change between two iterates that is still larger than a tolerance.
That last use is how every solution method of this course is written: iterate
until the answer stops moving, with an iteration cap as a safety net.

The recursion $F_{n+1} = \alpha + \alpha\beta F_n$ from "VFI by hand" is the
smallest example. It should converge to $F = \alpha / (1 - \alpha\beta)$ from any
$F_0$. Two Python details. A list grows with `append`, and the last element is
`dist[-1]`. Formatted printing uses f-strings, `f"{F:.8f}"`.

In [ ]:
F = 0.0                                      # the guess F_0 = 0, as on the slide
dist = [np.inf]                              # distances, one per iteration
n = 0
while dist[-1] > mpar.crit and n < mpar.maxiter:
    F_new = par.alpha + par.alpha * par.beta * F   # the recursion from "VFI by hand"
    dist.append(abs(F_new - F))              # record how much F moved
    F = F_new
    n += 1
print(f"after {n} iterations: F_n = {F:.8f}, the closed form F = {F_star:.8f}")

The contraction mapping theorem says the distance shrinks by a fixed factor per
step, here $\alpha\beta = 0.288$. Check it on the recorded distances: the ratio of
successive distances should settle on that number. A `for` loop, which runs a
fixed number of times, prints the first few.

In [ ]:
for i in range(2, 7):
    print(f"iteration {i}: distance {dist[i]:.2e}, ratio to the previous one {dist[i] / dist[i - 1]:.4f}")
print("alpha beta =", par.alpha * par.beta)

### 5. Plots

Three plots: the closed form value function on the grid, the closed form policy
against the 45 degree line with the steady state $k^*$ marked, and the
convergence of the recursion on a log scale, where geometric convergence is a
straight line. With Matplotlib, `plt.subplots` creates the axes, then `ax.plot` draws lines,
`ax.scatter` points, `ax.semilogy` a line on a log axis, and `ax.axhline` a
horizontal line. Labels go into `label=` and appear with `ax.legend()`.

In [ ]:
fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(14, 4))

ax1.plot(gri.k, v_grid, linewidth=2, label=r"$v(k) = E + F \ln k$")
ax1.set_xlabel("capital $k$")
ax1.set_title("Closed form value function")
ax1.legend(loc="lower right")

ax2.plot(gri.k, policy_grid, linewidth=2, label=r"$k' = \alpha\beta k^\alpha$")
ax2.plot(gri.k, gri.k, color="gray", linestyle="--", label="45 degree line")
ax2.scatter([kstar], [kstar], color="black", label="steady state $k^*$", zorder=5)
ax2.set_xlabel("capital today $k$")
ax2.set_ylabel("capital tomorrow $k'$")
ax2.set_title("Closed form policy")
ax2.legend(loc="upper left")

ax3.semilogy(range(1, n + 1), dist[1:], linewidth=2, label="$|F_n - F_{n-1}|$")
ax3.axhline(mpar.crit, color="gray", linestyle=":", label="tolerance")
ax3.set_xlabel("iteration $n$")
ax3.set_title("Convergence of the recursion")
ax3.legend()

fig.tight_layout()
plt.show()

### Exercises

1. In part 4, start from $F_0 = 40$ and from $F_0 = -5$. Does the limit change? Does the number of iterations?
2. Define CRRA utility $u(c) = (c^{1-\gamma} - 1)/(1-\gamma)$ as a second function with $\gamma = 2$ and plot both utility functions on a grid of consumption levels between $0.1$ and $2$. Where do they differ most?
3. Change the grid in part 1 to 20 points and to 500 points and rerun part 5. What changes in the plots, and what does not?